# 00 Full-scan findings, goals, and preflight

## Exact sequence in existing notebooks
1) Produce `coffea/<sample>.coffea` with `run_samp`/`remote_run_samp` using `vbfprocessor.VBFProcessor`.
2) Build WC map from reweight card + dictionary (`wc_map_dict`).
3) Extract HTXS yields for bin1=(221,222), bin2=(223,224).
4) Fit quadratic dependence in WC with `curve_fit`.
5) Convert yields to xsec using `start0_MG_sigma=3.594 pb`, `xsec_fb = 3.594*1000*yield/sumw_all_noEW`.
6) Build chi2 from observed values (for original Hbb-like setup in repo).
7) Scan WC grid and plot 1D/2D chi2.

## Our goals
- Use same EFT response machinery but replace measurement likelihood with HWW single μ measurement.
- First do HWW-only EFT fits.
- Then combine HWW chi2 with Hbb chi2 from repo logic.


## Notebook purpose
This notebook establishes the **analysis context** and verifies that all required inputs exist before any fit is attempted.

## Why this matters mathematically
All downstream fits assume a deterministic mapping: files → histograms → fitted coefficients → χ² scans. Missing or mismatched files invalidates the fit model.

## Code cell guide
- Import/setup cell: defines robust repo discovery and legacy module paths.
- Validation cell: checks listed source assets and file discoverability.
- Dependency cell: validates Python stack needed for numerical fitting and plotting.
- Summary cell: writes a compact machine-readable workflow summary for reproducibility.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from coffea import util

def find_repo_root(start=None, marker='smeft_contents.txt', max_up=8):
    p=Path(start or Path.cwd()).resolve()
    for _ in range(max_up+1):
        if (p/marker).exists(): return p
        if p.parent==p: break
        p=p.parent
    raise FileNotFoundError(f'Could not find {marker} from {Path.cwd()}')

REPO=find_repo_root()
NBDIR=REPO/'notebooks_hww_fit'
LISTING=REPO/'smeft_contents.txt'
for p in ['/uscms_data/d3/azhou/smeft/analysis','/uscms_data/d3/azhou/smeft/analysis/hbb-coffea']:
    if p not in sys.path: sys.path.append(p)
import stxs_functions as sf
print('repo:',REPO)
paths=[l.strip() for l in LISTING.read_text().splitlines() if l.strip()]
print('listing entries:',len(paths))
req=['/uscms_data/d3/azhou/smeft/analysis/stxs_functions.py','/uscms_data/d3/azhou/smeft/analysis/vbfprocessor.py']
for r in req: print(r, any(p==r or p.startswith(r+'/') for p in paths))

In [ ]:
mods=['numpy','scipy','matplotlib','awkward','uproot','hist','coffea']
for m in mods:
    try: __import__(m); print('ok',m)
    except Exception as e: print('missing',m,e)

In [ ]:
(NBDIR/'workflow_summary.json').write_text(json.dumps({'existing_sequence':'documented in notebook 00 markdown','goals':['HWW-only EFT fit','HWW+Hbb combination']},indent=2))